# Week 3, Lab 3 — Sequential vs hierarchical


In [ ]:
WEEK = 'Week 3'
LAB = 'Lab 3 — process types'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn crewai
else:
    %pip install -q crewai ollama


In [ ]:
cfg = openai_client_kwargs()
from crewai import LLM, Agent, Task, Crew, Process

llm = LLM(
    model=f"openai/{cfg['model']}",
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
)
print("CrewAI LLM ->", cfg)


In [ ]:
researcher = Agent(role="Researcher", goal="Collect 3 facts.", backstory="Analyst.", llm=llm)
writer = Agent(role="Writer", goal="Write a short paragraph.", backstory="Teacher.", llm=llm)
manager = Agent(role="Manager", goal="Delegate and deliver a clean student paragraph.", backstory="Project lead.", llm=llm)

t_research = Task(description="3 facts about Ollama.", expected_output="3 bullets", agent=researcher)
t_write = Task(description="One short paragraph for students using those facts.", expected_output="1 paragraph", agent=writer)

print("==== SEQUENTIAL ====")
print(Crew(agents=[researcher, writer], tasks=[t_research, t_write], process=Process.sequential).kickoff())

print("\n==== HIERARCHICAL ====")
print(Crew(
    agents=[researcher, writer, manager],
    tasks=[t_research, t_write],
    process=Process.hierarchical,
    manager_llm=llm,
).kickoff())


Hierarchical uses more tokens and can flake on tiny models — that is a reliability lesson. **Next:** tools.
